In [1]:
import torch

def generate_batch(batch_size, seq_len, vocab_size=10):
    X = torch.randint(0, vocab_size, (batch_size, seq_len))
    y = torch.flip(X, dims=[1])
    return X, y

In [2]:
X, y = generate_batch(batch_size=2, seq_len=5)
print(X)
print(y)


tensor([[6, 7, 7, 3, 9],
        [6, 5, 7, 6, 9]])
tensor([[9, 3, 7, 7, 6],
        [9, 6, 7, 5, 6]])


In [3]:
import torch.nn as nn
import math

class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = self._sinusoidal_encoding(max_seq_len, d_model)

    def _sinusoidal_encoding(self, max_seq_len, d_model):
        pos = torch.arange(max_seq_len).unsqueeze(1)
        i = torch.arange(d_model).unsqueeze(0)
        angle_rates = 1 / torch.pow(10000, (2 * (i // 2)) / d_model)
        angles = pos * angle_rates
        pe = torch.zeros(max_seq_len, d_model)
        pe[:, 0::2] = torch.sin(angles[:, 0::2])
        pe[:, 1::2] = torch.cos(angles[:, 1::2])
        return pe

    def forward(self, x):
        seq_len = x.shape[1]
        tok_emb = self.token_embed(x)
        pos_emb = self.pos_embed[:seq_len, :].unsqueeze(0)
        return tok_emb + pos_emb

In [4]:
embed_layer = TokenAndPositionEmbedding(vocab_size=10, max_seq_len=20, d_model=32)
X, y = generate_batch(batch_size=2, seq_len=5)
out = embed_layer(X)
print(out.shape)

torch.Size([2, 5, 32])


In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_model = d_model

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        weights = torch.softmax(scores, dim=-1)
        attn_output = weights @ V

        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output)

In [8]:
mha = MultiHeadAttention(d_model=32, num_heads=4)
out = mha(embed_layer(X))
print(out.shape)

torch.Size([2, 5, 32])


In [9]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out = self.attention(x)
        x = self.norm1(x + attn_out)
        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)
        return x

In [11]:
block = TransformerBlock(d_model=32, num_heads=4, d_ff=128)
out = block(embed_layer(X))
print(out.shape)

torch.Size([2, 5, 32])


In [12]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = TokenAndPositionEmbedding(vocab_size, max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x)

In [13]:
model = TinyTransformer(vocab_size=10, max_seq_len=20, d_model=32, num_heads=4, d_ff=128, num_layers=2)
X, y = generate_batch(batch_size=2, seq_len=5)
out = model(X)
print(out.shape)

torch.Size([2, 5, 10])


In [14]:
import torch.optim as optim

def train_model(model, epochs, batch_size, seq_len, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        X, y = generate_batch(batch_size, seq_len)

        logits = model(X)
        loss = loss_fn(logits.view(-1, logits.shape[-1]), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            predictions = logits.argmax(dim=-1)
            accuracy = (predictions == y).float().mean()
            print(f"Epoch {epoch}: loss = {loss.item():.4f}, accuracy = {accuracy.item():.4f}")

    return model

In [19]:
model = TinyTransformer(vocab_size=10, max_seq_len=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model = train_model(model, epochs=1000, batch_size=64, seq_len=6)

Epoch 0: loss = 2.4984, accuracy = 0.0651
Epoch 100: loss = 1.4896, accuracy = 0.4115
Epoch 200: loss = 0.0242, accuracy = 1.0000
Epoch 300: loss = 0.0094, accuracy = 1.0000
Epoch 400: loss = 0.0056, accuracy = 1.0000
Epoch 500: loss = 0.0037, accuracy = 1.0000
Epoch 600: loss = 0.0027, accuracy = 1.0000
Epoch 700: loss = 0.0020, accuracy = 1.0000
Epoch 800: loss = 0.0016, accuracy = 1.0000
Epoch 900: loss = 0.0013, accuracy = 1.0000


In [16]:
model.eval()
X_test, y_test = generate_batch(batch_size=64, seq_len=15)
with torch.no_grad():
    logits = model(X_test)
    predictions = logits.argmax(dim=-1)
    accuracy = (predictions == y_test).float().mean()
print(f"Accuracy on unseen length 15: {accuracy.item():.4f}")

Accuracy on unseen length 15: 0.1198


In [17]:
def evaluate_at_lengths(model, lengths, batch_size=200):
    model.eval()
    results = {}
    with torch.no_grad():
        for length in lengths:
            X_test, y_test = generate_batch(batch_size, length)
            logits = model(X_test)
            predictions = logits.argmax(dim=-1)
            accuracy = (predictions == y_test).float().mean().item()
            results[length] = accuracy
    model.train()
    return results

In [20]:
lengths_to_test = [6, 8, 10, 12, 15, 20, 25]
sinusoidal_results = evaluate_at_lengths(model, lengths_to_test)
print(sinusoidal_results)

{6: 1.0, 8: 0.21250000596046448, 10: 0.1655000001192093, 12: 0.24708333611488342, 15: 0.1080000028014183, 20: 0.1277499943971634, 25: 0.15320000052452087}


In [21]:
class TokenOnlyEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.token_embed(x)

In [22]:
class TinyTransformerNoPE(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = TokenOnlyEmbedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x)

In [23]:
model_nope = TinyTransformerNoPE(vocab_size=10, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_nope = train_model(model_nope, epochs=1000, batch_size=64, seq_len=6)

Epoch 0: loss = 2.4112, accuracy = 0.1016
Epoch 100: loss = 1.5845, accuracy = 0.3411
Epoch 200: loss = 1.4520, accuracy = 0.3594
Epoch 300: loss = 1.4779, accuracy = 0.3464
Epoch 400: loss = 1.3591, accuracy = 0.4062
Epoch 500: loss = 1.3952, accuracy = 0.3698
Epoch 600: loss = 1.3797, accuracy = 0.3828
Epoch 700: loss = 1.3742, accuracy = 0.3594
Epoch 800: loss = 1.3919, accuracy = 0.3307
Epoch 900: loss = 1.3485, accuracy = 0.3750


In [24]:
class LearnedPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)

    def forward(self, x):
        seq_len = x.shape[1]
        positions = torch.arange(seq_len).unsqueeze(0)
        return self.token_embed(x) + self.pos_embed(positions)


class TinyTransformerLearnedPE(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = LearnedPositionEmbedding(vocab_size, max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x)

In [25]:
model_learned = TinyTransformerLearnedPE(vocab_size=10, max_seq_len=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_learned = train_model(model_learned, epochs=1000, batch_size=64, seq_len=6)

learned_results = evaluate_at_lengths(model_learned, lengths_to_test)
print(learned_results)

Epoch 0: loss = 2.4007, accuracy = 0.1172
Epoch 100: loss = 0.1549, accuracy = 1.0000
Epoch 200: loss = 0.0239, accuracy = 1.0000
Epoch 300: loss = 0.0109, accuracy = 1.0000
Epoch 400: loss = 0.0064, accuracy = 1.0000
Epoch 500: loss = 0.0043, accuracy = 1.0000
Epoch 600: loss = 0.0031, accuracy = 1.0000
Epoch 700: loss = 0.0024, accuracy = 1.0000
Epoch 800: loss = 0.0018, accuracy = 1.0000
Epoch 900: loss = 0.0015, accuracy = 1.0000
{6: 1.0, 8: 0.25437501072883606, 10: 0.14100000262260437, 12: 0.17208333313465118, 15: 0.13733333349227905, 20: 0.12974999845027924, 25: 0.11860000342130661}
